#### Série: DuckDB Masterclass
**Ordem sugerida de leitura:** 6 de 14


📚 **Material de apoio — Laboratórios práticos no Jupyter Notebook**

Acompanhe os conceitos deste artigo com exemplos SQL, exercícios e laboratórios da DuckDB Masterclass.

🔗 [Acessar os laboratórios no GitHub](https://github.com/aestevaomoraes/duckdb-masterclass)


# DuckDB Masterclass — Do Básico ao Especialista
## Capítulo 06 — Junções temporais com ASOF JOIN e séries temporais
**Trilha:** Engenharia Analítica e Operations Analytics  
**Contexto:** consumo de dados, tarifas por vigência e auditoria operacional  
**Autor do material original:** Antonio Estevão  
**Edição:** revisão pedagógica e técnica; exemplos sintéticos e reproduzíveis.

> **Roteiro:** entender o problema → observar a linha do tempo → executar a junção → tratar exceções → consolidar faturamento → resolver desafio → validar domínio. Execute as células em ordem. Os exercícios não dependem de serviços pagos.

> **Trilha de mentoria:** execute o capítulo em ordem, registre hipóteses e valide resultados. Para uma aula completa, consulte a seção de atividades de reflexão ao fim.

**Nota:** este capítulo já havia sido revisado; a versão foi preservada e apenas complementada.

## 06.1 — Objetivos e pré-requisitos
Ao final você deverá conseguir:
1. Diferenciar `JOIN` por igualdade de um alinhamento temporal *as-of*.
2. Explicar por que `ASOF JOIN` retorna **no máximo um** registro do lado direito para cada evento do lado esquerdo.
3. Escolher entre `ASOF JOIN` e `ASOF LEFT JOIN` e auditar eventos sem tarifa.
4. Aplicar regras de vigência e calcular receita com segurança.
5. Tratar horários, `TIMESTAMP`, intervalos e empates de vigência.
6. Usar `EXPLAIN` para inspecionar um plano, sem pressupor complexidade ou velocidade.

**Pré-requisitos:** SELECT, JOIN, GROUP BY, TIMESTAMP e noções de granularidade. **Critério de domínio:** responder ao desafio final sem olhar o gabarito.

## 06.2 — O problema: eventos e tarifas em instantes diferentes
Um evento de consumo acontece às 14h30; a tarifa passou a vigorar no início do mês. Não há igualdade entre os instantes. Precisamos da **tarifa mais recente que já estava vigente** quando o evento ocorreu, mantendo a mesma chave `servico`.

**Linha do tempo (FIBRA):**

`01/jan — R$ 0,15/MB` → `10/jan — consumo 500 MB` → `01/fev — R$ 0,12/MB` → `15/fev — consumo 1.200 MB` → `01/mar — R$ 0,10/MB` → `20/mar — consumo 2.500 MB`

Para a condição `evento_ts >= inicio`, a junção escolhe a maior data `inicio` que não ultrapassa o evento. A condição de desigualdade é a essência da junção temporal. **Não é uma igualdade de timestamps**.

**Atenção à implementação:** o `ASOF JOIN` define a *semântica* do emparelhamento; não garante por si só busca binária, custo `O(N log M)` ou maior desempenho em toda situação. Avalie planos e dados reais.

## 06.3 — Preparação do laboratório
O setup utiliza uma conexão única em memória para todo o capítulo e dados determinísticos. Caso o ambiente não tenha as bibliotecas, instale separadamente com `pip install duckdb pandas` e reinicie o kernel.

In [ ]:
import duckdb
import pandas as pd
from IPython.display import display
print('DuckDB:', duckdb.__version__)
con = duckdb.connect(':memory:')

In [ ]:
con.execute("""
CREATE OR REPLACE TABLE precos_vigencia AS
SELECT * FROM (VALUES
 ('FIBRA', TIMESTAMP '2026-01-01 00:00:00', 0.15::DECIMAL(10,4)),
 ('FIBRA', TIMESTAMP '2026-02-01 00:00:00', 0.12::DECIMAL(10,4)),
 ('FIBRA', TIMESTAMP '2026-03-01 00:00:00', 0.10::DECIMAL(10,4)),
 ('MOVEL', TIMESTAMP '2026-02-01 00:00:00', 0.20::DECIMAL(10,4))
) AS t(servico, inicio, tarifa_mb);
CREATE OR REPLACE TABLE consumo_clientes AS
SELECT * FROM (VALUES
 (101, 'FIBRA', TIMESTAMP '2026-01-10 14:30:00', 500),
 (102, 'FIBRA', TIMESTAMP '2026-02-15 09:15:00', 1200),
 (103, 'FIBRA', TIMESTAMP '2026-03-20 18:00:00', 2500),
 (104, 'FIBRA', TIMESTAMP '2025-12-20 09:00:00', 100),
 (105, 'MOVEL', TIMESTAMP '2026-02-01 00:00:00', 100),
 (106, 'MOVEL', TIMESTAMP '2026-01-20 12:00:00', 200)
) AS t(cliente_id, servico, evento_ts, volume_mb);
""")
display(con.execute('SELECT * FROM precos_vigencia ORDER BY servico, inicio').df())
display(con.execute('SELECT * FROM consumo_clientes ORDER BY cliente_id').df())

## 06.4 — Laboratório A: ASOF JOIN (somente eventos com correspondência)
A junção considera a igualdade de `servico` e a desigualdade temporal `evento_ts >= inicio`. Entre os preços elegíveis, devolve o de início mais próximo ao evento, sem ultrapassá-lo. O `ASOF JOIN` padrão exclui eventos sem preço correspondente.

In [ ]:
res_asof = con.execute("""
SELECT 
        c.cliente_id, 
        c.servico, 
        c.evento_ts, 
        c.volume_mb,
        p.inicio AS vigencia_aplicada,
        p.tarifa_mb,
        ROUND(
            c.volume_mb * p.tarifa_mb, 2
        ) AS valor
FROM consumo_clientes AS c
ASOF JOIN precos_vigencia AS p
  ON c.servico = p.servico 
  AND c.evento_ts >= p.inicio
ORDER BY c.cliente_id
""").df()
display(res_asof)
assert res_asof['cliente_id'].tolist() == [101, 102, 103, 105]
assert [float(x) for x in res_asof['valor']] == [75.0, 144.0, 250.0, 20.0]
print('Validação do ASOF JOIN concluída.')

**Interprete antes de continuar:** por que os clientes 104 e 106 desapareceram? Eles registraram consumo **antes da primeira vigência cadastrada para seus serviços**. Não podemos tratá-los automaticamente como consumo zero: falta uma referência de preço.

## 06.5 — Laboratório B: preservar e auditar eventos sem preço
`ASOF LEFT JOIN` mantém todos os eventos do lado esquerdo. Quando não existe tarifa válida, as colunas da tarifa ficam `NULL`. Isso é essencial para auditoria e reconciliação.

In [ ]:
res_left = con.execute("""
SELECT 
    c.cliente_id,
    c.servico, 
    c.evento_ts, 
    c.volume_mb,
    p.inicio AS vigencia_aplicada, 
    p.tarifa_mb,
       ROUND(
           c.volume_mb * p.tarifa_mb, 2
        ) AS valor,
    
    CASE WHEN p.inicio IS NULL THEN 'SEM_TARIFA' ELSE 'TARIFADO' END AS status

FROM consumo_clientes c
ASOF LEFT JOIN precos_vigencia p
  ON c.servico = p.servico AND c.evento_ts >= p.inicio
ORDER BY c.cliente_id
""").df()
display(res_left)
assert len(res_left) == 6
assert set(res_left.loc[res_left['status']=='SEM_TARIFA','cliente_id']) == {104,106}
print('Auditoria: 6 eventos preservados; 2 sem tarifa.')

### Pontos de decisão profissional
- **Evento exatamente no início:** `>=` inclui a tarifa que começa naquele instante. `>` excluiria essa igualdade.
- **Múltiplos serviços:** é indispensável combinar a comparação temporal com `servico` para impedir tarifação cruzada.
- **Empates na tabela de preços:** dois preços com a **mesma chave e mesmo `inicio`** são ambíguos. Não dependa de qual registro o motor escolherá; valide e resolva o conflito antes da junção.
- **Fim da vigência:** o modelo aqui pressupõe validade até a próxima mudança. Se houver cancelamentos, janelas com lacunas ou término explícito, é preciso incluir essas regras separadamente.
- **Tempo civil:** timestamps sem fuso e instantes com fuso não são intercambiáveis; declare o fuso e padronize as fontes em projetos reais.

In [ ]:
duplicidades = con.execute("""
SELECT 
    servico,
    inicio,
    COUNT(*) AS n
FROM precos_vigencia
GROUP BY servico, inicio
HAVING COUNT(*) > 1
""").df()
assert duplicidades.empty, 'Há mais de uma tarifa iniciando no mesmo instante para o mesmo serviço.'
print('Qualidade: sem datas de início duplicadas por serviço.')

## 06.6 — Séries temporais: intervalos e agregações
A função de intervalo permite deslocar instantes; por exemplo, `TIMESTAMP '2026-02-01 00:00:00' + INTERVAL 30 DAY`. Para construir séries por dia/mês, primeiro defina a granularidade. Não confunda **soma por período** com **preço vigente por evento**.

In [ ]:
display(con.execute("""
SELECT cliente_id, evento_ts,
       evento_ts + INTERVAL 30 DAY AS revisao_em,
       date_trunc('month', evento_ts) AS mes_referencia
FROM consumo_clientes
ORDER BY cliente_id
""").df())

## 06.7 — Desafio guiado: faturamento determinístico de 1.000 eventos
O original simulava 1.000 eventos de consumo com números aleatórios. Aqui preservamos a escala, mas usamos volumes **reproduzíveis**, incluindo eventos após as mudanças de tarifa. Calcule a receita e a quantidade de eventos por vigência. Observe que volume de consumo não é a mesma coisa que uma chamada telefônica: nesta simulação, cada linha é um **evento de tráfego**.

In [ ]:
con.execute("""
CREATE OR REPLACE TABLE trafego_simulado AS
SELECT range AS id,
       'FIBRA' AS servico,
       TIMESTAMP '2026-01-01 00:00:00' + (range * INTERVAL 120 MINUTE) AS evento_ts,
       CAST(50 + (range % 10) * 25 AS INTEGER) AS volume_mb
FROM range(1000)
""")
print(con.execute('SELECT COUNT(*) FROM trafego_simulado').fetchone()[0], 'eventos')
display(con.execute('SELECT * FROM trafego_simulado ORDER BY id LIMIT 5').df())

### Sua missão (tente antes de executar a solução)
1. Relacione cada evento à tarifa vigente.
2. Conte eventos e some volume e receita **por início de vigência**.
3. Confira se nenhum evento ficou sem preço e se o total de eventos é 1.000.
4. Explique por que agregar apenas por serviço produz resposta diferente da agregação por vigência.

In [ ]:
# Solução de referência — consulte após tentar
res_faturamento = con.execute("""
WITH tarifacao AS (
 SELECT c.id, c.servico, c.evento_ts, c.volume_mb,
        p.inicio AS vigencia, p.tarifa_mb,
        CAST(c.volume_mb AS DECIMAL(18,2)) * p.tarifa_mb AS receita
 FROM trafego_simulado c
 ASOF LEFT JOIN precos_vigencia p
   ON c.servico = p.servico AND c.evento_ts >= p.inicio
)
SELECT servico, vigencia, COUNT(*) AS eventos,
       SUM(volume_mb) AS volume_total_mb,
       ROUND(SUM(receita), 2) AS receita_total
FROM tarifacao
GROUP BY servico, vigencia
ORDER BY servico, vigencia
""").df()
display(res_faturamento)
assert int(res_faturamento['eventos'].sum()) == 1000
assert not res_faturamento['vigencia'].isna().any()
assert len(res_faturamento) == 3
print('Validação: 1.000 eventos tarifados em três períodos de vigência.')

## 06.8 — Performance e plano de execução
Em bases volumosas, investigue cardinalidades, distribuição temporal, memória e plano de execução. `EXPLAIN` mostra o plano estimado; `EXPLAIN ANALYZE` executa a consulta e produz medições. Não existe promessa geral de complexidade `O(N log M)` ou ganho garantido do `ASOF JOIN`. Compare alternativas usando seus dados e sua versão do DuckDB.

In [ ]:
plano = con.execute("""
EXPLAIN SELECT c.id, p.tarifa_mb
FROM trafego_simulado c
ASOF LEFT JOIN precos_vigencia p
  ON c.servico = p.servico AND c.evento_ts >= p.inicio
""").fetchall()
for linha in plano:
    print(linha)

## 06.9 — Avaliação formativa (responda sem consultar as células)
**Q1.** Em `c.evento_ts >= p.inicio`, qual tarifa é selecionada?  
A) Qualquer tarifa anterior. B) A tarifa com maior `inicio` que não ultrapasse o evento. C) A primeira tarifa cadastrada.

**Q2.** Um evento sem tarifa deve permanecer no relatório de auditoria. Qual operador escolher?  
A) `ASOF JOIN`. B) `ASOF LEFT JOIN`. C) `CROSS JOIN`.

**Q3.** Há duas tarifas do mesmo serviço com o mesmo horário de início. O que fazer?  
A) Confiar na ordem física. B) Remover eventos. C) Aplicar regra de qualidade e resolver a ambiguidade.

**Q4.** Um evento acontece exatamente em `inicio`. Qual comparação inclui a nova tarifa?  
A) `>=`. B) `>`. C) `<`.

**Q5.** Por que não podemos prometer `O(N log M)` para todas as consultas `ASOF`?  
A) Porque o plano físico depende de versão, dados e otimizador. B) Porque o DuckDB não implementa joins. C) Porque tempos não são ordenáveis.

### Gabarito comentado
**1-B:** maior início elegível para a chave. **2-B:** preserva eventos sem correspondência com `NULL`. **3-C:** preços duplicados no mesmo instante representam regra ambígua. **4-A:** a igualdade inclui o instante inicial. **5-A:** semântica SQL não determina um custo assintótico único.

## 06.10 — Checklist de domínio e próximos passos
- [ ] Sei explicar a condição temporal e a direção da busca.
- [ ] Entendo a diferença de `ASOF JOIN` e `ASOF LEFT JOIN`.
- [ ] Sei identificar eventos sem tarifa e vigências duplicadas.
- [ ] Calculo faturamento por evento e agrego por vigência.
- [ ] Sei interpretar `INTERVAL`, `date_trunc` e a granularidade.
- [ ] Não confundo modelo lógico do join com performance garantida.

**Próxima aula:** executar os laboratórios, interpretar cada saída e discutir casos-limite. O gabarito fica no final para apoiar sua revisão.

**Referências oficiais:**
- [DuckDB — AsOf Join](https://duckdb.org/docs/stable/guides/sql_features/asof_join)
- [DuckDB — FROM and JOIN Clauses](https://duckdb.org/docs/stable/sql/query_syntax/from)
- [DuckDB — Date and Time Functions](https://duckdb.org/docs/stable/sql/functions/date)
- [DuckDB — EXPLAIN](https://duckdb.org/docs/stable/guides/meta/explain)

## 🔎 Avaliação de especialista — reflexão antes da próxima aula

1. Qual decisão de negócio o laboratório permite tomar?
2. Que hipótese a consulta assume sobre os dados?
3. Que teste automatizado detectaria uma violação dessa hipótese?
4. Como explicar o resultado sem consultar o código?

Anote as respostas em uma nova célula Markdown.